In [1]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [2]:
from pathlib import Path

img = Path(r"C:\Users\Admin\Desktop\Grad_Project\Code\Datasets\bdd100k_images_100k\100k\train")
lbl = Path(r"C:\Users\Admin\Desktop\Grad_Project\Code\Datasets\BDD100k_V2\labels\train")

imgs = list(img.glob("*.jpg"))
lbls = list(lbl.glob("*.txt"))

print(f"Images: {len(imgs):,}")
print(f"Labels: {len(lbls):,}")
print(f"Sample image: {imgs[0].stem}")
print(f"Matching label exists: {(lbl / (imgs[0].stem + '.txt')).exists()}")

Images: 70,000
Labels: 70,000
Sample image: 0000f77c-6257be58
Matching label exists: True


In [3]:
"""
Tamakkan - YOLOv11s Training Script
=====================================
Dataset  : BDD100K full 100k, 7 classes
Model    : YOLOv11s (small, real-time capable on-device)
Target   : mAP50 >= 0.90 on dominant classes, >= 0.75 on VRU/bus
 
Class imbalance strategy:
  - cls_pw (classification pos weight) boosted for rare classes
  - copy_paste augmentation to synthetically boost VRU instances
  - mosaic=1.0 forces rare classes into more training images
  - cos_lr for smooth convergence over long training
"""
 
from ultralytics import YOLO
from pathlib import Path
import torch

In [4]:
# ── VERIFY GPU ─────────────────────────────────────────────────────────────────
assert torch.cuda.is_available(), "CUDA not available — check PyTorch installation"
print(f"Training on: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB\n")

Training on: NVIDIA GeForce RTX 5080
VRAM: 17.1 GB



In [5]:
# ── PATHS ──────────────────────────────────────────────────────────────────────
DATA_YAML  = r"C:\Users\Admin\Desktop\Grad_Project\Code\Datasets\BDD100k_V2\data.yaml"
OUTPUT_DIR = r"C:\Users\Admin\Desktop\Grad_Project\Code\Yolov11s_training_Results"

In [6]:
# ── LOAD MODEL ─────────────────────────────────────────────────────────────────
# Loads YOLOv11s pretrained on COCO — we fine-tune from here, not scratch
model = YOLO("yolo11s.pt")

In [7]:
# ── TRAIN ──────────────────────────────────────────────────────────────────────
results = model.train(
    # --- Core ---
    data        = DATA_YAML,
    epochs      = 100,
    imgsz       = 640,
    batch       = 16,          # safe for 16GB VRAM with YOLOv11s at 640px
    device      = 0,           # RTX 5080
 
    # --- Output ---
    project     = OUTPUT_DIR,
    name        = "tamakkan_v1",
    exist_ok    = False,
 
    # --- Optimization ---
    optimizer   = "AdamW",
    lr0         = 0.001,       # initial LR
    lrf         = 0.01,        # final LR = lr0 * lrf
    momentum    = 0.937,
    weight_decay= 0.0005,
    cos_lr      = True,        # cosine LR schedule — smooth convergence
    warmup_epochs    = 3,
    warmup_momentum  = 0.8,
 
    # --- Imbalance handling ---
    # cls_pw: upweights the positive classification loss for rare classes.
    # car is 48x more common than VRU — this pushes the model to care more
    # about getting rare classes right instead of just predicting car everywhere
    cls         = 0.7,         # classification loss weight (default 0.5, raised slightly)
 
    # --- Augmentation (key for rare class boosting) ---
    mosaic      = 1.0,         # always on — composites 4 images, rare classes appear more
    copy_paste  = 0.3,         # copies rare object instances into other images
    mixup       = 0.1,         # blends two images — helps generalization
    degrees     = 10.0,        # rotation augmentation
    translate   = 0.1,
    scale       = 0.5,
    shear       = 2.0,
    perspective = 0.0,
    flipud      = 0.01,
    fliplr      = 0.5,
    hsv_h       = 0.015,
    hsv_s       = 0.7,
    hsv_v       = 0.4,
 
    # --- Early stopping ---
    patience    = 20,          # stop if no improvement for 20 epochs
 
    # --- Efficiency ---
    workers     = 4,           # dataloader workers
    cache       = False,       # don't cache — 100k images is too large to cache in RAM
    amp         = True,        # mixed precision — faster training on RTX 5080
    close_mosaic= 10,          # disable mosaic in final 10 epochs for stable convergence
 
    # --- Logging ---
    plots       = True,        # save training plots
    save        = True,
    save_period = 10,          # save checkpoint every 10 epochs
    verbose     = True,
)
 
print("\nTraining complete.")
print(f"Results saved to: {OUTPUT_DIR}")
print(f"Best weights: {OUTPUT_DIR}/tamakkan_v1/weights/best.pt")

Ultralytics 8.4.37  Python-3.12.10 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5080, 16303MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.7, cls_pw=0.0, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:\Users\Admin\Desktop\Grad_Project\Code\Datasets\BDD100k_V2\data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.01, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=tamakkan_v17, nbs=64, nms=False, opset=None, op